### 第一部分实验 测试不同的评价指标

In [4]:
import os
import sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

dataset_path = os.path.join(os.path.abspath('..'), 'training')
sys.path.append(dataset_path)

import cv2
import json
import yaml
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy import ndimage
from PIL import Image
from sklearn.manifold import TSNE
from torchvision import transforms
from dataset import *
from detectors import DETECTOR
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Subset

from scipy.fft import fft2, fftshift
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from scipy.spatial.distance import pdist

In [ ]:
def prepare_training_data(config):
    train_set = DeepfakeAbstractBaseDataset(
        config=config,
        mode='train',
    )

    train_dataloader = \
        torch.utils.data.DataLoader(
            dataset=train_set,
            batch_size=config['train_batchSize'],
            shuffle=False,
            num_workers=int(config['workers']),
            collate_fn=train_set.collate_fn,
            drop_last=False
            )
    return train_set, train_dataloader

def init_seed(config):
    if config['manualSeed'] is None:
        config['manualSeed'] = random.randint(1, 10000)
    random.seed(config['manualSeed'])
    if config['cuda']:
        torch.manual_seed(config['manualSeed'])
        torch.cuda.manual_seed_all(config['manualSeed'])

def show_gray_image(img_array):
    """显示标准化后的RGB图像及其灰度图（适用于(224,224,1,3)形状）"""
    # 调整维度：移除通道组维度（假设形状为(H,W,1,3)）
    img_standard = img_array.squeeze(axis=2)  # 转为(H,W,3)
    
    # RGB转灰度（标准权重）
    gray_img = 0.299 * img_standard[..., 0] + 0.587 * img_standard[..., 1] + 0.114 * img_standard[..., 2]
    
    # 显示图像
    plt.figure(figsize=(10, 4))
    plt.subplot(121), plt.imshow(img_standard), plt.title('原始RGB'), plt.axis('off')
    plt.subplot(122), plt.imshow(gray_img, cmap='gray'), plt.title('黑白图'), plt.axis('off')
    plt.tight_layout(), plt.show()

def split_image_lists(image_paths):
    """
    将图像路径列表按类别拆分为5个子列表：origin、F2F、NT、DF、FS
    
    参数:
    image_paths (list): 包含所有图像路径的列表
    
    返回:
    tuple: 包含5个子列表的元组，顺序为(origin, F2F, NT, DF, FS)
    """
    origin = []  # 原始图像
    F2F = []     # Face2Face 伪造图像
    NT = []      # NeuralTextures 伪造图像
    DF = []      # Deepfakes 伪造图像
    FS = []      # FaceSwap 伪造图像
    
    for path in image_paths:
        # 检查是否为原始图像
        if "original_sequences" in path:
            origin.append(path)
        # 检查是否为Face2Face伪造图像
        elif "Face2Face" in path:
            F2F.append(path)
        # 检查是否为NeuralTextures伪造图像
        elif "NeuralTextures" in path:
            NT.append(path)
        # 检查是否为Deepfakes伪造图像
        elif "Deepfakes" in path:
            DF.append(path)
        # 检查是否为FaceSwap伪造图像
        elif "FaceSwap" in path:
            FS.append(path)
    return origin, F2F, NT, DF, FS

In [ ]:
with open('/data/yyl/model/deepfake/DeepfakeBench-main/training/config/detector/clip3.yaml', 'r') as f:
    config = yaml.safe_load(f)
with open('/data/yyl/model/deepfake/DeepfakeBench-main/training/config/train_config.yaml', 'r') as f:
    config2 = yaml.safe_load(f)
config.update(config2)
init_seed(config)

# 获取数据集
train_dataset, train_dataloader = prepare_training_data(config)
origin, F2F, NT, DF, FS = split_image_lists(train_dataset.image_list)
dataset_dict = {'origin': origin, 'F2F': F2F, 'NT': NT, 'DF': DF, 'FS': FS}

image_dict = {}
for key, val in dataset_dict.items():
    for item in val:
        image_dict[item] = [key]

### 验证面部分块傅里叶变换差异性


In [ ]:
def process_image(image_path, block_size=16):
    """处理单张图像，返回统计特征"""
    file_path = f'{config["rgb_dir"]}/'+image_path.replace('\\', '/')
    img = cv2.imread(file_path)
    if img is None:
        print(f"警告: 无法读取图像 {file_path}")
        return (file_path, None)  # 返回元组，包含图像路径和处理结果
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    feats = []
    for i in range(64, h-64, block_size):
        for j in range(64, w-64, block_size):
            blk = gray[i:i+block_size, j:j+block_size]
            if blk.shape == (block_size, block_size):
                F = fftshift(fft2(blk))
                feats.append(np.log1p(np.abs(F)).flatten())

    if len(feats) < 2:
        print(f"警告: 图像 {file_path} 的有效特征块少于2个")
        return (file_path, None)

    dists = pdist(feats, metric='euclidean')
    return (image_path, [dists.mean(), dists.max(), dists.std(), dists.var()])
    # return (image_path, {
    #     'mean': dists.mean(),
    #     'max':  dists.max(),
    #     'std':  dists.std(),
    #     'var':  dists.var()
    # })

In [ ]:
# 准备要处理的图像路径列表
image_paths = list(image_dict.keys())

# 使用多进程池处理图像
with Pool(processes=30) as pool:
    # imap 方法返回一个迭代器，可与 tqdm 结合显示进度
    results = list(tqdm(
        pool.imap(process_image, image_paths),
        total=len(image_paths),
        desc="Processing images"
    ))

# 更新 image_dict，过滤掉 None 值
valid_results = 0
invalid_results = 0

for result in results:
    if result is None:
        invalid_results += 1
        continue  # 跳过 None 值
    
    image_path, features = result
    if features is not None:
        image_dict[image_path].extend(features)
        valid_results += 1
    else:
        invalid_results += 1

In [ ]:
data = []
for image_path, values in image_dict.items():
    if len(values) >= 5:
        data.append({
            'image_path': image_path,
            'class': values[0],  # 类别
            'fft_mean': values[1],  # 均值
            'fft_max': values[2],   # 最大值
            'fft_std': values[3],   # 标准差
            'fft_var': values[4]    # 方差
        })

df = pd.DataFrame(data)
required_columns = ['image_path', 'class', 'fft_mean', 'fft_max', 'fft_std', 'fft_var']
df = df[required_columns]
df.to_excel('fft.xlsx', index=False)

# 颜色空间一致性

In [7]:
def calculate_channel_noise(channel, block_size=8):
    """计算单个通道的噪声方差"""
    # 局部均值滤波估计信号
    local_mean = ndimage.gaussian_filter(channel, sigma=1.0)
    
    # 计算原始块方差和信号方差
    h, w = channel.shape
    noise_vars = []
    
    for i in range(0, h - block_size + 1, block_size):
        for j in range(0, w - block_size + 1, block_size):
            # 提取块
            block = channel[i:i+block_size, j:j+block_size]
            block_mean = local_mean[i:i+block_size, j:j+block_size]
            
            # 估计噪声方差 (块方差 - 信号方差)
            block_var = np.var(block)
            signal_var = np.var(block_mean)
            
            # 确保噪声方差非负
            noise_var = max(0, block_var - signal_var)
            noise_vars.append(noise_var)
    
    return np.array(noise_vars)

def process_image_ycbcr(image_path, block_size=8):
    """处理单张图像，计算YCbCr空间噪声一致性评分"""
    try:
        # 读取图像
        file_path = f'{config["rgb_dir"]}/'+image_path.replace('\\', '/')
        img = cv2.imread(file_path)
        if img is None:
            print(f"警告: 无法读取图像 {file_path}")
            return (file_path, None)
        
        # 转换到YCbCr颜色空间
        ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)  # OpenCV顺序是YCrCb而不是YCbCr
        
        # 分离通道
        Y = ycbcr[:,:,0]
        Cr = ycbcr[:,:,1]
        Cb = ycbcr[:,:,2]
        
        # 计算每个通道的噪声方差
        noise_vars_Y = calculate_channel_noise(Y, block_size)
        noise_vars_Cr = calculate_channel_noise(Cr, block_size)
        noise_vars_Cb = calculate_channel_noise(Cb, block_size)
        
        # 确保所有通道的噪声方差数量相同（可能由于边缘处理导致差异）
        min_len = min(len(noise_vars_Y), len(noise_vars_Cr), len(noise_vars_Cb))
        noise_vars_Y = noise_vars_Y[:min_len]
        noise_vars_Cr = noise_vars_Cr[:min_len]
        noise_vars_Cb = noise_vars_Cb[:min_len]
        
        # 计算通道间噪声方差差异
        diff_Y_Cr = np.abs(noise_vars_Y - noise_vars_Cr)
        diff_Y_Cb = np.abs(noise_vars_Y - noise_vars_Cb)
        diff_Cr_Cb = np.abs(noise_vars_Cr - noise_vars_Cb)
        
        # 计算评分统计量
        mean_diff_Y_Cr = np.mean(diff_Y_Cr)
        mean_diff_Y_Cb = np.mean(diff_Y_Cb)
        mean_diff_Cr_Cb = np.mean(diff_Cr_Cb)
        
        std_diff_Y_Cr = np.std(diff_Y_Cr)
        std_diff_Y_Cb = np.std(diff_Y_Cb)
        std_diff_Cr_Cb = np.std(diff_Cr_Cb)
        
        # 综合评分 (可以根据需要调整权重)
        score = (mean_diff_Y_Cr + mean_diff_Y_Cb + mean_diff_Cr_Cb) / 3
        
        # 返回详细评分结果
        return (image_path, {
            'score': score,
            'mean_diff_Y_Cr': mean_diff_Y_Cr,
            'mean_diff_Y_Cb': mean_diff_Y_Cb,
            'mean_diff_Cr_Cb': mean_diff_Cr_Cb,
            'std_diff_Y_Cr': std_diff_Y_Cr,
            'std_diff_Y_Cb': std_diff_Y_Cb,
            'std_diff_Cr_Cb': std_diff_Cr_Cb,
            'num_blocks': min_len
        })
    
    except Exception as e:
        print(f"处理图像 {image_path} 时出错: {str(e)}")
        return (image_path, None)

# 定义一个全局函数替代lambda函数
def process_wrapper(args):
    return process_image_ycbcr(*args)

def batch_process_images(image_paths, block_size=8, num_processes=4):
    """批量处理图像"""
    results = []
    
    # 创建参数列表
    args_list = [(path, block_size) for path in image_paths]
    
    with Pool(processes=num_processes) as pool:
        # 使用全局函数替代lambda函数
        for result in tqdm(
            pool.imap(process_wrapper, args_list),
            total=len(image_paths),
            desc="处理图像"
        ):
            results.append(result)
    
    return results

def save_results_to_dataframe(results):
    """将结果保存到DataFrame"""
    import pandas as pd
    
    data = []
    for image_path, features in results:
        if features is not None:
            data.append({
                'image_path': image_path,
                'noise_score': features['score'],
                'mean_diff_Y_Cr': features['mean_diff_Y_Cr'],
                'mean_diff_Y_Cb': features['mean_diff_Y_Cb'],
                'mean_diff_Cr_Cb': features['mean_diff_Cr_Cb'],
                'std_diff_Y_Cr': features['std_diff_Y_Cr'],
                'std_diff_Y_Cb': features['std_diff_Y_Cb'],
                'std_diff_Cr_Cb': features['std_diff_Cr_Cb'],
                'num_blocks': features['num_blocks']
            })
    
    df = pd.DataFrame(data)
    return df

In [11]:
image_paths = list(image_dict.keys())
results = batch_process_images(image_paths, block_size=16, num_processes=20)
df = save_results_to_dataframe(results)
df.to_csv('./Ycrcb.csv')
df.head()

处理图像: 100%|██████████| 114884/114884 [07:07<00:00, 268.96it/s]


,image_path,noise_score,mean_diff_Y_Cr,mean_diff_Y_Cb,mean_diff_Cr_Cb,std_diff_Y_Cr,std_diff_Y_Cb,std_diff_Cr_Cb,num_blocks
0,FaceForensics++\original_sequences\youtube\c23...,86.637008,129.416985,129.634113,0.859927,207.637126,207.527091,1.391152,256
1,FaceForensics++\original_sequences\youtube\c23...,42.433887,62.270798,63.579756,1.451108,127.219425,128.167702,2.537699,256
2,FaceForensics++\original_sequences\youtube\c23...,55.398812,82.805540,82.354906,1.035989,133.021394,132.953869,1.819467,256
3,FaceForensics++\original_sequences\youtube\c23...,48.237979,65.622083,68.189122,10.902730,109.760692,107.643555,31.139345,256
4,FaceForensics++\original_sequences\youtube\c23...,45.919604,67.750741,66.691647,3.316426,130.930447,131.784775,8.743956,256


# 保存评分

In [12]:
df_ycbcr = pd.read_csv('/data/yyl/model/deepfake/DeepfakeBench-main/AAAI/Ycrcb.csv')
df_score = pd.read_csv('//data/yyl/model/deepfake/DeepfakeBench-main/AAAI/scores.csv')

In [14]:
prefix = "/data/yyl/data/deepfake_dataset/rgb/"
df_ycbcr['processed_name'] = df_ycbcr['image_path'].str.replace(prefix, '', regex=False)
df_ycbcr['processed_name'] = df_ycbcr['processed_name'].str.replace('/', '\\')

# 2. 合并数据（仅保留prediction列）
merged_df = pd.merge(
    df_score, 
    df_ycbcr[['processed_name', 'noise_score']], 
    left_on='image_path', 
    right_on='processed_name',
    how='left'
)

# 3. 重新排列列顺序，将prediction列添加到df_fft原有列之后
original_columns = df_score.columns.tolist()
new_columns = original_columns + ['noise_score']
merged_df = merged_df[new_columns]
merged_df.to_csv('./scores1.csv', index=False)